In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ranjeeetz/sih26-ps45-syn-data/ip_rag_classifier_1000_observations.csv


In [3]:
!pip list | grep -E 'numpy|pandas|sklearn'

geopandas                                1.1.3
numpy                                    2.0.2
pandas                                   2.3.3
pandas-datareader                        0.10.0
pandas-gbq                               0.30.0
pandas-profiling                         3.6.6
pandas-stubs                             2.2.2.240909
pandasql                                 0.7.3
sklearn-compat                           0.1.5
sklearn-pandas                           2.2.0


In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report

In [6]:
df = pd.read_csv('/kaggle/input/datasets/ranjeeetz/sih26-ps45-syn-data/ip_rag_classifier_1000_observations.csv',index_col='observation_id')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 1 to 1000
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   subject_matter          964 non-null    object
 1   primary_objective       931 non-null    object
 2   technical_invention     968 non-null    object
 3   brand_identifier        965 non-null    object
 4   product_appearance      976 non-null    object
 5   geographical_origin     931 non-null    object
 6   creative_expression     926 non-null    object
 7   bio_or_plant_matter     287 non-null    object
 8   confidentiality         923 non-null    object
 9   regulated_product_type  396 non-null    object
 10  target_category         1000 non-null   object
dtypes: object(11)
memory usage: 93.8+ KB


In [8]:
for col in df.columns:
  print(f'Column: {col}')
  print(df[col].unique())
  print('-' * 40)

Column: subject_matter
['Biological/Plant Matter' 'General Regulation' 'Confidential Information'
 'Product' 'Regulated Product' 'Brand' 'Creative Work' 'Invention' 'Other'
 nan]
----------------------------------------
Column: primary_objective
['Get Protection' nan 'Registration' 'Enforcement/Dispute'
 'Compliance/Approval' 'Other' 'Commercialization']
----------------------------------------
Column: technical_invention
['Unclear' 'No' 'Yes' nan]
----------------------------------------
Column: brand_identifier
['No' 'Unclear' nan 'Yes']
----------------------------------------
Column: product_appearance
['No' 'Yes' 'Unclear' nan]
----------------------------------------
Column: geographical_origin
['No' 'Unclear' 'Yes' nan]
----------------------------------------
Column: creative_expression
['No' nan 'Yes' 'Unclear']
----------------------------------------
Column: bio_or_plant_matter
['Plant Variety' nan 'Traditional Knowledge' 'Biological Resource']
------------------------------

In [10]:
model = CalibratedClassifierCV(estimator=Pipeline([
    ('ohe',OneHotEncoder(handle_unknown='infrequent_if_exist',drop='first',min_frequency=1e-2,sparse_output=False)),
    ('hgbm',HistGradientBoostingClassifier(random_state=34))
],verbose=False,memory=None),
method='isotonic',cv=5,n_jobs=-1,ensemble=True)

In [12]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['target_category']),df['target_category'],test_size=0.3,random_state=34)

In [13]:
model.fit(X_train,y_train)

CalibratedClassifierCV(cv=5, ensemble=True,
                       estimator=Pipeline(steps=[('ohe',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='infrequent_if_exist',
                                                                min_frequency=0.01,
                                                                sparse_output=False)),
                                                 ('hgbm',
                                                  HistGradientBoostingClassifier(random_state=34))]),
                       method='isotonic', n_jobs=-1)

In [14]:
print(classification_report(model.predict(X_test),y_test))

                          precision    recall  f1-score   support

          Ayurveda-Aahar       0.97      0.97      0.97        29
    Biological Diversity       1.00      1.00      1.00        23
               Copyright       0.96      0.88      0.92        25
                  Design       1.00      0.96      0.98        28
         Drug Regulation       1.00      0.95      0.98        22
         Food Regulation       0.97      1.00      0.98        29
 Geographical Indication       1.00      1.00      1.00        16
                  Patent       0.96      1.00      0.98        24
Plant Variety Protection       1.00      1.00      1.00        25
              Regulatory       1.00      1.00      1.00        29
            Trade Secret       1.00      1.00      1.00        22
               Trademark       0.93      1.00      0.97        28

                accuracy                           0.98       300
               macro avg       0.98      0.98      0.98       300
        

In [15]:
import joblib

joblib.dump(model,'IPShaktiClassification.joblib')

['IPShaktiClassification.joblib']

In [16]:
the_model = joblib.load('IPShaktiClassification.joblib')

In [18]:
# Define test cases mapping to your 12 target categories
test_data = {
    'subject_matter': [
        'Invention', 'Brand', 'Creative Work', 'Confidential Information', 
        'Product', 'Regulated Product', 'Regulated Product', 'Other', 
        'Biological/Plant Matter', 'Biological/Plant Matter', 'Regulated Product', 'General Regulation'
    ],
    'primary_objective': [
        'Get Protection', 'Registration', 'Get Protection', 'Enforcement/Dispute', 
        'Registration', 'Compliance/Approval', 'Compliance/Approval', 'Get Protection', 
        'Compliance/Approval', 'Get Protection', 'Compliance/Approval', 'Compliance/Approval'
    ],
    'technical_invention': ['Yes', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No'],
    'brand_identifier': ['No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No'],
    'product_appearance': ['No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No'],
    'geographical_origin': ['No', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No'],
    'creative_expression': ['No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No'],
    'bio_or_plant_matter': [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 'Biological Resource', 'Plant Variety', np.nan, np.nan],
    'confidentiality': ['No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No'],
    'regulated_product_type': [np.nan, np.nan, np.nan, np.nan, np.nan, 'Food', 'Ayurveda-Aahar', np.nan, np.nan, np.nan, 'Drug', np.nan]
}

# Expected corresponding target categories for validation
expected_categories = [
    'Patent', 
    'Trademark', 
    'Copyright', 
    'Trade Secret', 
    'Design', 
    'Food Regulation', 
    'Ayurveda-Aahar', 
    'Geographical Indication', 
    'Biological Diversity', 
    'Plant Variety Protection', 
    'Drug Regulation', 
    'Regulatory'
]

In [19]:
# Create DataFrame
X_new = pd.DataFrame(test_data)

# Predict
predictions = the_model.predict(X_new)

# Display results comparison
results_df = pd.DataFrame({
    'Expected': expected_categories,
    'Predicted': predictions
})
print(results_df)

                    Expected                 Predicted
0                     Patent                    Patent
1                  Trademark                 Trademark
2                  Copyright                 Copyright
3               Trade Secret              Trade Secret
4                     Design                    Design
5            Food Regulation           Food Regulation
6             Ayurveda-Aahar            Ayurveda-Aahar
7    Geographical Indication                Regulatory
8       Biological Diversity      Biological Diversity
9   Plant Variety Protection  Plant Variety Protection
10           Drug Regulation           Drug Regulation
11                Regulatory                Regulatory
